# 2. Basics of Optimization 
## Part B Discrete optimization

Let's have a sandwich structure of length $L$ consisting of $N$ blocks with the same thickness.
Each block can be either air, acoustic foam, or melamine foam.

Your task is to find the best sequence of materials to maximize absorption.

![Geometry](optB.png)




In [ ]:
# import necessary libraries
import numpy as np
import matplotlib.pyplot as plt

from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.soo.nonconvex.ga import GA
from pymoo.optimize import minimize

from optimization_utils import *

Define variables needed for the problem:

In [ ]:
N_BLOCKS = 10 # play around with this number to see how it affects the optimization results

block_length = L / N_BLOCKS
lengths_b = [block_length] * N_BLOCKS
# print(f"Block lengths: {lengths_b}")

PALETTE = [ 
    MATERIALS["air"],
    MATERIALS["acoustic_foam"],
    MATERIALS["melamine_foam"],
]

N_MATS = len(PALETTE)

Optimization itself:

In [ ]:
# 1. Define discrete problem with integer variable type
class DiscreteSandwichProblem(ElementwiseProblem):
    def __init__(self):
        super().__init__(
            n_var=N_BLOCKS,
            n_obj=1,
            xl=0,
            xu=N_MATS - 1,
            vtype=int  # Tells pymoo to handle discrete integer operators
        )

    def _evaluate(self, x, out, *args, **kwargs):
        mat_list = [PALETTE[int(idx)] for idx in x]
        reflection, absorption = compute_spectrum(lengths_b, mat_list)

        # obj = -np.mean(absorption)            # Minimize negative mean absorption
        # obj = np.mean(np.abs(reflection))     # Does minimizing the mean reflection coefficient give the same result?
        obj = -np.mean(absorption[freqs<300]) # Minimize negative mean absorption but only for frequencies below 300 Hz
        # obj = -np.max(absorption)             # Minimize negative max absorption

        out["F"] = obj

# 2. Minimal GA setup
problem = DiscreteSandwichProblem()
algorithm = GA(pop_size=40)

res = minimize(
    problem,
    algorithm,
    termination=('n_gen', 30),
    seed=42,
    verbose=True # enables logging of the optimization process
)

# 3. Extract and display best sequence
best_sequence = res.X.astype(int)
best_materials = [PALETTE[idx] for idx in best_sequence]

In [ ]:
print("Optimal Material Sequence (Front to Back Wall):")
print(" | ".join([m.name for m in best_materials]))

In [ ]:
# Reflection and absorption for optimal case and single-material baselines
r_opt, abs_opt = compute_spectrum(lengths_b, best_materials)
matA = MATERIALS["acoustic_foam"]
matB = MATERIALS["melamine_foam"]
r_matA, abs_matA = compute_spectrum([L], [matA])
r_matB, abs_matB = compute_spectrum([L], [matB])

# Plot results
plt.figure(figsize=(15, 5))
plt.subplot(121)
plt.plot(freqs, np.abs(r_opt), 'k-', linewidth=2, label='Part B: optimized')
plt.plot(freqs, np.abs(r_matA), '--', label=f'Only {matA.name}')
plt.plot(freqs, np.abs(r_matB), '--', label=f'Only {matB.name}')
setup_r_axis()

plt.subplot(122)
plt.plot(freqs, abs_opt, 'k-', linewidth=2, label='Part B: optimized')
plt.plot(freqs, abs_matA, '--', label=f'Only {matA.name}')
plt.plot(freqs, abs_matB, '--', label=f'Only {matB.name}')
setup_abs_axis()
plt.show()
